# Chapter 1 &mdash; Regular Patterns: Repetition Without Counting

**Concept 11 of the Chapter 1 decomposition:** *Pattern Class I — Regular Patterns*

Keywords, password rules, identifiers, comma lists. Unbounded repetition is fine; <b>remembering how much you repeated</b> is not.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter1-Intro/Concept-Regular-Patterns/Concept-Regular-Patterns.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.LangDef        import *
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.AnimateDFA     import *

import jove; print('Jove loaded from', list(jove.__path__)[0])
import jove.AnimateDFA as _a; print('animation toolbar:',
      'ready' if hasattr(_a.AnimateDFA, '_ipython_display_')
      else 'STALE -- restart the runtime, then re-run')

## 1. The idea


Two flavours:

* **finite and fixed-size** &mdash; the keyword `main` is four letters in that order;
* **finite but unbounded** &mdash; an identifier is a letter then any number of
  letters/digits; `keyword id, id, id;` is a keyword, then any number of ID+comma
  pairs, then `;`.

Regular patterns may repeat forever but **cannot count**. That single limitation is
what separates them from the next class.

## 2. Definitions

### The comma-list pattern as a DFA

`keyword id, id, id;` &mdash; here `k` is keyword, `i` is id, `c` is comma, `s` is semicolon.

In [ ]:
commalist = md2mc('''DFA
I   : k -> Ak
Ak  : i -> Ai
Ai  : c -> Ak
Ai  : s -> F
F   : k | i | c | s -> BH
I   : i | c | s -> BH
Ak  : k | c | s -> BH
Ai  : k | i -> BH
BH  : k | i | c | s -> BH
''')
print("comma-list DFA states :", sorted(commalist["Q"]))

### A "boring" repetition

$01001010010100101001\ldots$ is just an alternation of `01` and `001`. Boring is the
point: regular patterns repeat without memory of how often.

In [ ]:
boring = lstar({'01', '001'}, 3)
print("some strings over {01, 001} :", sorted(boring, key=len)[:10])

<!-- nav-strip -->

---

&larr;&nbsp;[Ch1&nbsp;10.&nbsp;The Unboundedness Principle: Design as if There Were No Limits](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter1-Intro/Concept-Unboundedness-Principle/Concept-Unboundedness-Principle.ipynb) &nbsp;&middot;&nbsp; [**Chapter 1** index](https://github.com/ganeshutah/Jove/blob/master/Chapter1-Intro/README.md) &nbsp;&middot;&nbsp; [Ch1&nbsp;12.&nbsp;Pattern Class II — Context-Free Patterns](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter1-Intro/Concept-Context-Free-Patterns/Concept-Context-Free-Patterns.ipynb)&nbsp;&rarr;

---

## 3. Tests

Any number of ID+comma pairs is accepted &mdash; **unbounded, but never counted**.

In [ ]:
for n in range(1, 7):
    s = 'k' + 'ic' * (n-1) + 'is'
    print("%d ids : %-18s accepted? %s" % (n, s, accepts_dfa(commalist, s)))
assert all(accepts_dfa(commalist, "k" + "ic"*(n-1) + "is") for n in range(1, 20))

Malformed declarations are rejected &mdash; the book's `keyword ; id id id,,` case.

In [ ]:
for s in ['ksiiicc', 'kis', 'kiciis', 'ks', 'kicis']:
    print("%-10s accepted? %s" % (s, accepts_dfa(commalist, s)))
assert not accepts_dfa(commalist, "ksiiicc")

The machine stays the **same size** however long the input is. That is the signature
of a regular pattern.

In [ ]:
print("states in the DFA        :", len(commalist["Q"]))
print("longest input we accepted:", len('k' + 'ic'*49 + 'is'))
print("accepted?                :", accepts_dfa(commalist, 'k' + 'ic'*49 + 'is'))
print()
print("50 ids, still 6 states. The machine never counted them.")

## 4. Animation


Watch the DFA cycle between "expecting an id" and "expecting a comma or semicolon".
The cycle is the unbounded repetition.

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(commalist, FuseEdges=True)

## 5. Exercises


1. Design a DFA for the password rule "between 4 and 12 characters". How many states?
   Why is this still *finite and fixed-size*?
2. Modify `commalist` to allow an **empty** declaration `keyword ;`.
3. Try to design a DFA for "the same number of `(` as `)`". Where does it go wrong?
   (That is Concept 11.)

In [ ]:
# Your work for the exercises above.

## 6. Where next

In [ ]:
# Previous / next, and a search box for all 253 concepts.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter1-Intro/Concept-Regular-Patterns')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')